# LIBERO — **덜 학습된 체크포인트 제거 + 재학습**

상태표에서 **목표 step(150k) 미달** 체크포인트를 찾아 제거하고, 미학습분과 함께 다시 학습한다.

- **UNDER**(0<step<150k, 예: 70k·80k) → 폴더 제거 후 처음부터 재학습.
- **MISSING**(체크포인트 없음, 예: acm·mosaic) → 신규 학습.
- **done**(≥150k) → 손 안 댐. eval 은 150k 체크포인트로 하므로 200k 까지 간 것도 그대로 OK.
- `bimamba_s7`/`mosaic` 은 config 키가 달라 매핑을 걸어줌(아래 셀에서 정책 확인).
- ⚠️ 순서: **① 상태 → ② 제거(EXECUTE) → ③ 학습**. 두 노드면 각 노드에서 `NODE_IDX` 만 바꿔 실행.


In [ ]:
import sys, shutil
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

TASK   = 'libero_10'
SEEDS  = [0, 1, 2, 3]
TARGET = cf.CKPT_STEP                    # 150,000 — 이보다 낮으면 '학습 덜 됨'. eval 도 150k 로 함.

# ── folder_tag → 학습 config 키 (make_train_cmd 가 정책/lr/flags 를 여기서 읽음) ──
#   bimamba_s7(ours)  = acm2 + carry + BiMamba + overlap(추론)  → bimamba_mosaic
#   mosaic(mosaic only)= acm2 + carry + overlap(추론), BiMamba 없음 → mosaic_infer
#   ※ ours=bimamba+mosaic, mosaic only=ours−bimamba 로 강제되는 매핑. 팀 확인 권장.
TAG_CFG = {
    'act': 'act', 'acm': 'acm', 'acm2': 'acm2', 'bimamba': 'bimamba',
    'bimamba_s7': 'bimamba_mosaic',
    'mosaic': 'mosaic_infer',
}
for folder, cfgkey in TAG_CFG.items():
    cf.v23.MODEL_DIR_NAMES.setdefault(folder, folder)          # 출력 폴더 = folder 이름
    if folder not in cf.v23.MODEL_CONFIGS:
        cf.v23.MODEL_CONFIGS[folder] = cf.v23.MODEL_CONFIGS[cfgkey]   # 정책은 cfgkey 것 사용

print('학습 설정 확인 (folder → policy / flags):')
for folder, cfgkey in TAG_CFG.items():
    pol, lr, K, extra, cp = cf.v23.MODEL_CONFIGS[folder]
    fl = ' '.join(x for x in extra if 'sscp' in x or 'bimamba' in x or 'overlap' in x)
    print(f'   {folder:12} → {cfgkey:16} {pol:38} {fl}')
print(f'\n목표 step = {TARGET:,} (이보다 낮은 체크포인트 = 재학습 대상)')

## 1) 상태 판단 — done / UNDER(제거+재학습) / MISSING(학습)


In [ ]:
# ── 각 (모델,seed) 학습 상태: done(>=목표) / under(0<step<목표) / missing(없음) ──
def tstep(tag, s):
    return cf.v23.last_ckpt_step(cf.v23.train_dir(tag, s, TASK))

print(f"{'model':<14}{'seed':>5}{'last ckpt':>12}   상태")
print('-' * 42)
done, under, missing = [], [], []
for folder in TAG_CFG:
    for s in SEEDS:
        step = tstep(folder, s)
        if step is None:
            missing.append((folder, s)); state = 'MISSING → 학습'
        elif step < TARGET:
            under.append((folder, s)); state = f'UNDER → 제거+재학습'
        else:
            done.append((folder, s)); state = 'done'
        print(f"{folder:<14}{s:>5}{(f'{step:,}' if step else '-'):>12}   {state}")

print('\n' + '=' * 42)
print('done           :', len(done))
print('UNDER(제거+재학습):', [f'{t}/s{s}' for t, s in under] or '없음')
print('MISSING(학습)   :', [f'{t}/s{s}' for t, s in missing] or '없음')
jobs_needed = [(t, s, TASK) for (t, s) in under + missing]
print(f'\n→ 학습해야 할 (모델,seed): {len(jobs_needed)}개')

## 2) UNDER 체크포인트 제거 (dry-run → EXECUTE=True)
**done 폴더는 절대 안 지운다.** UNDER 만. 확인 후 `EXECUTE=True`.


In [ ]:
# ── UNDER(덜 학습된) 체크포인트 폴더 제거 ── 확인 후 EXECUTE=True 로 다시 실행 ──
EXECUTE = False

targets = [cf.v23.train_dir(t, s, TASK) for (t, s) in under]
if not targets:
    print('제거할 UNDER 폴더 없음 (덜 학습된 체크포인트 없음).')
else:
    print(f'{"제거 실행" if EXECUTE else "DRY-RUN(아무것도 안 지움)"} — UNDER {len(targets)}개:')
    for (t, s), d in zip(under, targets):
        step = cf.v23.last_ckpt_step(d)
        print(f'   {t}/seed{s}  ({step:,} step)  {d}')
        if EXECUTE and Path(d).is_dir():
            shutil.rmtree(d)
    if EXECUTE:
        print('\n제거 완료. 이제 아래 학습 셀 실행 → 처음부터 다시 학습.')
    else:
        print('\n확인됐으면 EXECUTE=True 로 바꿔 다시 실행. (done 폴더는 절대 안 건드림)')

## 3) 재학습/신규학습 (GPU 4+4 분할, 목표 150k)


In [ ]:
# ── 재학습/신규학습 ── UNDER(제거됨)+MISSING 를 목표 step 까지. GPU 4+4 두 노드로 분할 ──
GPU_COUNTS = [4, 4]     # 노드별 GPU. 한 노드만 쓰면 [8] 또는 실제 수, NODE_IDX=0
NODE_IDX   = 0         # 이 노드 번호(0=첫째, 1=둘째)

mine = cf.split_by_gpu(jobs_needed, GPU_COUNTS)[NODE_IDX]
gpus = cf.available_gpus()[:GPU_COUNTS[NODE_IDX]]
print(f'노드 {NODE_IDX+1} | GPU {gpus} | 이 노드 학습 {len(mine)}개 (전체 {len(jobs_needed)})')
for t, s, _ in mine:
    print(f'   {t:12} seed{s}')
print()
# 목표 step 까지 학습(이미 목표 도달분은 자동 skip, prefetch 포함, 데이터셋 레이스 방지)
cf.run_training_jobs(mine, gpus=gpus, prefetch_task=TASK)

## 4) 다음 단계


In [ ]:
# ── 다음 단계 안내 ──
print('학습이 끝나면:')
print('  1) libero/eval_node1_4gpu · eval_node2_4gpu 로 재학습분 500ep eval (RECORD_DIR 로 action 저장)')
print('  2) libero/eval_final 로 최종 SR·떨림 표 (500ep + action 있는 것만)')
print('\n※ 재학습한 (모델,seed) 의 옛 eval 결과(덜 학습된 체크포인트로 낸 것)는')
print('   eval_clean/libero_10/<model>/seed<N> 에서 지워야 새 eval 로 덮인다(필요시).')